# 07 - LightGBM (extra ML model)
Adds a row to `outputs/leakfree_results.csv`. Needs notebook 05 output. `pip install lightgbm` if missing.

In [1]:
import sys, os
from pathlib import Path
# works whether Jupyter starts in the project root or inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))      # lets us import data_utils.py from the root
print("Project root:", ROOT)

Project root: c:\Users\aru12\OneDrive\Desktop\capstone project


## 1. Imports and data

In [2]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from lightgbm import LGBMClassifier
from data_utils import *

(ROOT / "models").mkdir(exist_ok=True)
(Xtr, ytr), (Xva, yva), (Xte, yte) = load_splits()
print(len(Xtr), len(Xva), len(Xte))

389196 131775 236927


## 2. Build and train (target encoder is fitted on train only, inside the Pipeline)

In [3]:
pipe = Pipeline([
    ("p", ColumnTransformer([("te", TargetEncoder(target_type="binary", cv=5, random_state=SEED), FEATURES)])),
    ("m", LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
                         subsample=0.8, subsample_freq=1, colsample_bytree=0.9, reg_lambda=2.0,
                         class_weight="balanced", random_state=SEED, n_jobs=-1, verbose=-1))])
pipe.fit(Xtr, ytr)

c:\Users\aru12\anaconda3\envs\chicago-crime\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('p', ...), ('m', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['Primary Type','Location Description','Beat']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('te', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers

## 3. Evaluate (threshold from validation, scores on test) and save

In [4]:
t = best_threshold(yva, pipe.predict_proba(Xva)[:, 1])
p_te = pipe.predict_proba(Xte)[:, 1]
report("LightGBM", yte, p_te, t)
print("test ROC-AUC:", round(roc_auc_score(yte, p_te), 4))
joblib.dump(pipe, ROOT / "models/LightGBM.joblib")
save_rows([metrics_row("LightGBM", yte, p_te, t)])
pd.read_csv(RESULTS)

LightGBM               thr=0.55  accuracy=0.8302  balanced_acc=0.8301  gap=0.0001
test ROC-AUC: 0.9114


,Model,Features,Train/Val/Test,Test_rows,Threshold,Accuracy,Balanced_Accuracy,Precision,Recall,F1,ROC_AUC
0,MLP (embeddings),Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.57,0.8327,0.8336,0.5397,0.8350,0.6556,0.9136
1,Wide&Deep (DL),Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.58,0.8332,0.8320,0.5408,0.8301,0.6549,0.9127
2,LogisticRegression,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.53,0.8271,0.8276,0.5299,0.8285,0.6463,0.9094
3,RandomForest,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.51,0.8304,0.8301,0.5357,0.8295,0.6510,0.9104
4,HistGradientBoosting,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.55,0.8296,0.8299,0.5342,0.8303,0.6501,0.9113
5,XGBoost,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.52,0.8312,0.8311,0.5371,0.8309,0.6524,0.9125
6,LightGBM,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.55,0.8302,0.8301,0.5353,0.8299,0.6508,0.9114
